In [ ]:
!pip install -q transformers==4.48.3
!pip install -q datasets==3.2.0
!pip install -q peft==0.14.0
!pip install -q trl==0.15.2
!pip install -q bitsandbytes==0.45.2
!pip install -q accelerate==1.3.0
!pip install -q scikit-learn

In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTTrainer, SFTConfig

from sklearn.metrics import accuracy_score

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

OUTPUT_DIR = "sentiment_qlora_model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!


In [ ]:
dataset = load_dataset("glue", "sst2")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
print(dataset["train"][0])
print(dataset["train"][1])
print(dataset["train"][2])

{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
{'sentence': 'contains no wit , only labored gags ', 'label': 0, 'idx': 1}
{'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1, 'idx': 2}


In [ ]:
train_data = dataset["train"].shuffle(
    seed=42
).select(range(1000))

test_raw = dataset["validation"].shuffle(
    seed=42
).select(range(100))

print("Training:", len(train_data))
print("Testing:", len(test_raw))

Training: 1000
Testing: 100


In [ ]:
train_data = train_data.map(
    format_data,
    remove_columns=train_data.column_names
)

test_data = test_raw.map(
    format_data,
    remove_columns=test_raw.column_names
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
def format_data(example):

    label = "Positive" if example["label"] == 1 else "Negative"

    messages = [
        {
            "role": "system",
            "content": "You are a sentiment classification assistant. Reply with only Positive or Negative."
        },
        {
            "role": "user",
            "content": f"Classify this review: {example['sentence']}"
        },
        {
            "role": "assistant",
            "content": label
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {"text": text}

# The mapping was already successfully completed in the previous cell (eD9i8xDoayzU).
# We just print the first example from the already transformed train_data.
print(train_data[0]["text"])

<|im_start|>system
You are a sentiment classification assistant. Reply with only Positive or Negative.<|im_end|>
<|im_start|>user
Classify this review: klein , charming in comedies like american pie and dead-on in election , <|im_end|>
<|im_start|>assistant
Positive<|im_end|>



In [ ]:
# Load the model without 4-bit quantization when GPU is not available
try:
    # Try loading with 4-bit quantization (requires GPU)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
    print("4-bit model loaded successfully on GPU!")
except Exception as e:
    print(f"Could not load with GPU quantization due to: {e}")
    print("Attempting to load on CPU without quantization...")
    # Fallback to standard CPU loading
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="cpu",
        torch_dtype=torch.float32
    )
    print("Model loaded successfully on CPU!")

model.config.use_cache = False

CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend


Could not load with GPU quantization due to: CUDA is required but not available for bitsandbytes. Please consider installing the multi-platform enabled version of bitsandbytes, which is currently a work in progress. Please check currently supported platforms and installation instructions at https://huggingface.co/docs/bitsandbytes/main/en/installation#multi-backend
Attempting to load on CPU without quantization...
Model loaded successfully on CPU!


In [ ]:
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(

    r=16,

    lora_alpha=32,

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("LoRA configuration created!")

LoRA configuration created!


In [ ]:
training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="epoch",

    fp16=True,

    gradient_checkpointing=True,

    optim="paged_adamw_8bit",

    lr_scheduler_type="cosine",

    warmup_steps=10,

    max_seq_length=256,

    dataset_text_field="text",

    packing=False,

    report_to="none"
)

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduced batch size for CPU
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,                     # Disabled FP16 (requires GPU)
    use_cpu=True,                   # Force CPU usage
    gradient_checkpointing=True,
    optim="adamw_torch",            # Standard CPU-compatible optimizer
    lr_scheduler_type="cosine",
    warmup_steps=10,
    max_seq_length=256,
    dataset_text_field="text",
    packing=False,
    report_to="none"
)

In [ ]:
import re

def predict_sentiment(model, review):

    messages = [
        {
            "role": "system",
            "content": "You are a sentiment classification assistant. Reply with only Positive or Negative."
        },
        {
            "role": "user",
            "content": f"Classify this review: {review}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    match = re.search(
        r"\b(positive|negative)\b",
        generated,
        re.IGNORECASE
    )

    if match:
        return match.group(1).capitalize()

    return "Unknown"

In [ ]:
# Evaluate the base model
base_predictions = []

for review in test_raw["sentence"]:

    prediction = predict_sentiment(
        model,
        review
    )

    base_predictions.append(prediction)


true_labels = [
    "Positive" if label == 1 else "Negative"
    for label in test_raw["label"]
]

base_accuracy = accuracy_score(
    true_labels,
    base_predictions
)

print("Base Model Accuracy:", base_accuracy * 100)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Base Model Accuracy: 87.0


In [ ]:
# Evaluate the fine tuned model
ft_model = trainer.model

ft_predictions = []

for review in test_raw["sentence"]:

    prediction = predict_sentiment(
        ft_model,
        review
    )

    ft_predictions.append(prediction)


ft_accuracy = accuracy_score(
    true_labels,
    ft_predictions
)

print("Fine-tuned Model Accuracy:", ft_accuracy * 100)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:650: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/l

Fine-tuned Model Accuracy: 87.0


In [ ]:
print("BASE MODEL ACCURACY:",
      round(base_accuracy * 100, 2), "%")

print("FINE-TUNED MODEL ACCURACY:",
      round(ft_accuracy * 100, 2), "%")

improvement = (
    ft_accuracy - base_accuracy
) * 100

print("Improvement:",
      round(improvement, 2), "percentage points")

BASE MODEL ACCURACY: 87.0 %
FINE-TUNED MODEL ACCURACY: 87.0 %
Improvement: 0.0 percentage points


In [ ]:
errors = []

for i in range(len(test_raw)):

    if ft_predictions[i] != true_labels[i]:

        errors.append({

            "Review": test_raw[i]["sentence"],

            "Actual": true_labels[i],

            "Predicted": ft_predictions[i]

        })


print("Total errors:", len(errors))

for i, error in enumerate(errors[:3]):

    print("\nFailure Example:", i + 1)

    print("Review:", error["Review"])

    print("Actual:", error["Actual"])

    print("Predicted:", error["Predicted"])

Total errors: 13

Failure Example: 1
Review: you 'll gasp appalled and laugh outraged and possibly , watching the spectacle of a promising young lad treading desperately in a nasty sea , shed an errant tear . 
Actual: Positive
Predicted: Negative

Failure Example: 2
Review: fresnadillo 's dark and jolting images have a way of plying into your subconscious like the nightmare you had a week ago that wo n't go away . 
Actual: Positive
Predicted: Negative

Failure Example: 3
Review: true tale of courage -- and complicity -- at auschwitz is a harrowing drama that tries to tell of the unspeakable . 
Actual: Positive
Predicted: Negative


In [ ]:
# Check whether fine-tuning actually changed predictions
changed = 0

for i in range(len(base_predictions)):

    if base_predictions[i] != ft_predictions[i]:
        changed += 1

print("Predictions changed:", changed)

print("Base accuracy:", base_accuracy * 100)

print("Fine-tuned accuracy:", ft_accuracy * 100)

trainer.model.print_trainable_parameters()

Predictions changed: 0
Base accuracy: 87.0
Fine-tuned accuracy: 87.0
trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
for item in trainer.state.log_history:

    if "loss" in item:
        print(item["loss"])

**CONCLUSION**


## 2. Analysis of Three Failure Examples

### Objective
The task requires us to identify three examples where the fine-tuned model makes incorrect predictions and explain why these errors might have occurred.

### Failure Example 1

**Review:**  
"you 'll gasp appalled and laugh outraged and possibly , watching the spectacle of a promising young lad treading desperately in a nasty sea , shed an errant tear ."

**Actual Sentiment:** Positive  
**Predicted Sentiment:** Negative

**Reason for Failure:**  
The model may have focused on negative words such as "appalled" and "outraged" instead of understanding the overall meaning of the review.

### Failure Example 2

**Review:**  
"fresnadillo 's dark and jolting images have a way of plying into your subconscious like the nightmare you had a week ago that wo n't go away ."

**Actual Sentiment:** Positive  
**Predicted Sentiment:** Negative

**Reason for Failure:**  
The model may have interpreted words such as "dark" and "nightmare" as negative, without understanding that the review may be praising the movie's atmosphere and impact.

### Failure Example 3

**Review:**  
"true tale of courage -- and complicity -- at auschwitz is a harrowing drama that tries to tell of the unspeakable ."

**Actual Sentiment:** Positive  
**Predicted Sentiment:** Negative

**Reason for Failure:**  
The model may have focused on emotionally negative words such as "harrowing" and "unspeakable" rather than recognizing that the review may be appreciating the powerful storytelling.

### Conclusion

These errors suggest that the model sometimes relies too much on individual negative words instead of understanding the overall context and meaning of a sentence.
